In [3]:
import pandas as pd
import numpy as np
import os


In [5]:
os.getcwd()

'D:\\D.S. Programme\\BDM Project\\My Project'

In [4]:
!pip install xlrd
!pip install openpyxl



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Cleaning data from 202404 to 202503

In [72]:
df = pd.read_excel('project_data_202404_202503.XLS')

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8923 entries, 0 to 8922
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              3459 non-null   object 
 1   Bill              8228 non-null   object 
 2   Item Description  8574 non-null   object 
 3   Batch             7877 non-null   object 
 4   Qty.              0 non-null      float64
 5   Free              8575 non-null   float64
 6   Rate              8227 non-null   float64
 7   Amount            8575 non-null   float64
dtypes: float64(4), object(4)
memory usage: 557.8+ KB


In [69]:
df.head(20)

,Date,Bill,Item Description,Batch,Qty.,Free,Rate,Amount
0,01-04-2024,000001,DISPO. 3PLY,ELAS,NaN,35.0,85.7160,3000.06
1,01-04-2024,000001,DISPO. 3PLY,ELAS,NaN,240.0,75.0000,18000.00
2,01-04-2024,000001,SHOE COVER,PLAST,NaN,60.0,160.0000,9600.00
3,01-04-2024,000001,SAMA GUAZE,SMDGS3801,NaN,1800.0,11.1625,20092.50
4,01-04-2024,000001,ARC 4'*3.6M,23072502,NaN,260.0,95.0000,24700.00
5,01-04-2024,000001,ARC 3'*3.6M,23072502,NaN,201.0,85.0000,17085.00
6,01-04-2024,000001,CAP BUFFERANT,NaN,NaN,160.0,52.0000,8320.00
7,01-04-2024,000001,CISLON PLUS,100M AL2403050,NaN,200.0,9.0000,1800.00
8,01-04-2024,000001,CISLON PLUS,1000 AL2403046,NaN,45.0,44.0000,1980.00
9,01-04-2024,000001,HANDLOOM GAUGE,S 054,NaN,500.0,14.2880,7144.00


In [70]:
import pandas as pd
import numpy as np
import re

# Load your raw file
#df = pd.read_csv("your_file.csv")  # or pd.read_excel("your_file.xlsx")

# Make a copy
df_raw = df.copy()

# 1. Drop summary rows and blank rows
def is_summary_row(row):
    return pd.isna(row['Bill']) and isinstance(row['Date'], (int, float))

df = df[~df.apply(is_summary_row, axis=1)]  # Remove summary rows
df = df.dropna(how='all')  # Remove fully blank rows

# 2. Fix misaligned rows where Date is blank but Bill is actually a date
def fix_misaligned(row):
    if pd.isna(row['Date']) and pd.notna(row['Bill']):
        # Shift Date from Bill
        row['Date'] = row['Bill']
        if pd.notna(row['Item Description']):
            parts = str(row['Item Description']).split()
            if len(parts) == 1:
                row['Bill'] = parts[0]
                row['Item Description'] = ''
            if len(parts) > 1:
                row['Bill'] = parts[0]
                row['Item Description'] = ' '.join(parts[1:])
        else:
            row['Bill'] = np.nan
            row['Item Description'] = ''
    return row

df = df.apply(fix_misaligned, axis=1)

# 3. Clean Batch and Item Description logic
def clean_batch_and_description(row):
    batch = str(row['Batch']).strip() if pd.notna(row['Batch']) else ''
    desc = str(row['Item Description']).strip() if pd.notna(row['Item Description']) else ''
    
    batch_chunks = batch.split()
    
    if len(batch_chunks) > 1:
        row['Batch'] = batch_chunks[-1].strip()
        row['Item Description'] = desc + ' ' + ' '.join(chunk.strip() for chunk in batch_chunks[:-1])
    
    elif len(batch_chunks) == 1:
        chunk = batch_chunks[0].strip()
        
        # If it's all digits, treat as valid batch
        if chunk.isdigit():
            return row
        
        is_soft_descriptor = re.fullmatch(r"[A-Za-z().\-*/'/]+", chunk)
        is_known_unit = re.fullmatch(r"\d+[*xX]?\d*(GM|ML|KG|MG|L|TAB|CAP|UNIT|FT|IN)?", chunk, re.IGNORECASE)
        is_feet_inches = re.fullmatch(r"\d{1,3}['\"]", chunk)  # matches 6', 8", etc.
        
        if is_soft_descriptor or is_known_unit or is_feet_inches:
            row['Item Description'] = desc + ' ' + chunk
            row['Batch'] = ''
    
    return row



df = df.apply(clean_batch_and_description, axis=1)

# 4. Clean columns
df['Free'] = df['Free'].fillna(0)

# Forward fill Date
df['Date'] = df['Date'].fillna(method='ffill')

# Convert types
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
for col in ['Qty.', 'Free', 'Rate', 'Amount']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Optional: Reset index and strip strings
df = df.reset_index(drop=True)
df['Item Description'] = df['Item Description'].str.strip()
df['Batch'] = df['Batch'].str.strip()
df['Bill'] = df['Bill'].astype(str).str.strip()
df['Date'] = df['Date'].dt.strftime('%d-%m-%Y')  # Optional formatting

# Save to cleaned file



C:\Users\pranj\AppData\Local\Temp\ipykernel_11744\2816488763.py:74: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Date'] = df['Date'].fillna(method='ffill')


In [71]:
df.to_excel('project_data_202404_202503_cleaned_13.xlsx', index=False) 

### Cleaning data from 202304 to 202403

In [6]:
os.getcwd()

'D:\\D.S. Programme\\BDM Project\\My Project'

In [11]:
df = pd.read_excel(r'project_data_202304_202403.xlsx')

In [12]:
df.head(200)

,Date,Bill,Item Description,Batch,Qty.,Free,Rate,Amount
0,01-04-2023,2,HART MAEIN,FORCI,NaN,10.0,300.0000,3000
1,01-04-2023,2,"TOWEL CLIP 4""",NaN,NaN,15.0,160.0000,2400
2,01-04-2023,2,CROSACTION,TOWEL,NaN,35.0,40.0000,1400
3,01-04-2023,2,BOB COCK 8',NaN,NaN,15.0,235.0000,3525
4,01-04-2023,2,KOCHER ARTERY,8',NaN,30.0,235.0000,7050
...,...,...,...,...,...,...,...,...
195,NaN,06-04-2023,A232400431 3004 MEDICINE,300,NaN,1.0,6134.8000,6134.8
196,2,NaN,06-04-2023,NaN,NaN,2.0,NaN,23774.8
197,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,07-04-2023,SMC00046,RMS SCALP VEIN,S G221210875,NaN,8000.0,4.4135,35308


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11834 entries, 0 to 11833
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              7041 non-null   object 
 1   Bill              11120 non-null  object 
 2   Item Description  11408 non-null  object 
 3   Batch             10705 non-null  object 
 4   Qty.              0 non-null      float64
 5   Free              11476 non-null  float64
 6   Rate              11119 non-null  float64
 7   Amount            11476 non-null  object 
dtypes: float64(3), object(5)
memory usage: 739.8+ KB


In [14]:
import pandas as pd
import numpy as np
import re

# Load your raw file
#df = pd.read_csv("your_file.csv")  # or pd.read_excel("your_file.xlsx")

# Make a copy
df_raw = df.copy()

# 1. Drop summary rows and blank rows
def is_summary_row(row):
    return pd.isna(row['Bill']) and isinstance(row['Date'], (int, float))

df = df[~df.apply(is_summary_row, axis=1)]  # Remove summary rows
df = df.dropna(how='all')  # Remove fully blank rows

# 2. Fix misaligned rows where Date is blank but Bill is actually a date
def fix_misaligned(row):
    if pd.isna(row['Date']) and pd.notna(row['Bill']):
        # Shift Date from Bill
        row['Date'] = row['Bill']
        if pd.notna(row['Item Description']):
            parts = str(row['Item Description']).split()
            if len(parts) == 1:
                row['Bill'] = parts[0]
                row['Item Description'] = ''
            if len(parts) > 1:
                row['Bill'] = parts[0]
                row['Item Description'] = ' '.join(parts[1:])
        else:
            row['Bill'] = np.nan
            row['Item Description'] = ''
    return row

df = df.apply(fix_misaligned, axis=1)

# 3. Clean Batch and Item Description logic
def clean_batch_and_description(row):
    batch = str(row['Batch']).strip() if pd.notna(row['Batch']) else ''
    desc = str(row['Item Description']).strip() if pd.notna(row['Item Description']) else ''
    
    batch_chunks = batch.split()
    
    if len(batch_chunks) > 1:
        row['Batch'] = batch_chunks[-1].strip()
        row['Item Description'] = desc + ' ' + ' '.join(chunk.strip() for chunk in batch_chunks[:-1])
    
    elif len(batch_chunks) == 1:
        chunk = batch_chunks[0].strip()
        
        # If it's all digits, treat as valid batch
        if chunk.isdigit():
            return row
        
        is_soft_descriptor = re.fullmatch(r"[A-Za-z().\-*/'/]+", chunk)
        is_known_unit = re.fullmatch(r"\d+[*xX]?\d*(GM|ML|KG|MG|L|TAB|CAP|UNIT|FT|IN)?", chunk, re.IGNORECASE)
        is_feet_inches = re.fullmatch(r"\d{1,3}['\"]", chunk)  # matches 6', 8", etc.
        
        if is_soft_descriptor or is_known_unit or is_feet_inches:
            row['Item Description'] = desc + ' ' + chunk
            row['Batch'] = ''
    
    return row



df = df.apply(clean_batch_and_description, axis=1)

# 4. Clean columns
df['Free'] = df['Free'].fillna(0)

# Forward fill Date
df['Date'] = df['Date'].fillna(method='ffill')

# Convert types
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
for col in ['Qty.', 'Free', 'Rate', 'Amount']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Optional: Reset index and strip strings
df = df.reset_index(drop=True)
df['Item Description'] = df['Item Description'].str.strip()
df['Batch'] = df['Batch'].str.strip()
df['Bill'] = df['Bill'].astype(str).str.strip()
df['Date'] = df['Date'].dt.strftime('%d-%m-%Y')  # Optional formatting

# Save to cleaned file



C:\Users\pranj\AppData\Local\Temp\ipykernel_16696\2816488763.py:74: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Date'] = df['Date'].fillna(method='ffill')


In [15]:
df.to_excel('project_data_202304_202403_cleaned.xlsx', index=False) 

### Cleaning data from 202204 to 202303

In [16]:
df = pd.read_excel(r'project_data_202204_202303.xlsx')

In [17]:
import pandas as pd
import numpy as np
import re

# Load your raw file
#df = pd.read_csv("your_file.csv")  # or pd.read_excel("your_file.xlsx")

# Make a copy
df_raw = df.copy()

# 1. Drop summary rows and blank rows
def is_summary_row(row):
    return pd.isna(row['Bill']) and isinstance(row['Date'], (int, float))

df = df[~df.apply(is_summary_row, axis=1)]  # Remove summary rows
df = df.dropna(how='all')  # Remove fully blank rows

# 2. Fix misaligned rows where Date is blank but Bill is actually a date
def fix_misaligned(row):
    if pd.isna(row['Date']) and pd.notna(row['Bill']):
        # Shift Date from Bill
        row['Date'] = row['Bill']
        if pd.notna(row['Item Description']):
            parts = str(row['Item Description']).split()
            if len(parts) == 1:
                row['Bill'] = parts[0]
                row['Item Description'] = ''
            if len(parts) > 1:
                row['Bill'] = parts[0]
                row['Item Description'] = ' '.join(parts[1:])
        else:
            row['Bill'] = np.nan
            row['Item Description'] = ''
    return row

df = df.apply(fix_misaligned, axis=1)

# 3. Clean Batch and Item Description logic
def clean_batch_and_description(row):
    batch = str(row['Batch']).strip() if pd.notna(row['Batch']) else ''
    desc = str(row['Item Description']).strip() if pd.notna(row['Item Description']) else ''
    
    batch_chunks = batch.split()
    
    if len(batch_chunks) > 1:
        row['Batch'] = batch_chunks[-1].strip()
        row['Item Description'] = desc + ' ' + ' '.join(chunk.strip() for chunk in batch_chunks[:-1])
    
    elif len(batch_chunks) == 1:
        chunk = batch_chunks[0].strip()
        
        # If it's all digits, treat as valid batch
        if chunk.isdigit():
            return row
        
        is_soft_descriptor = re.fullmatch(r"[A-Za-z().\-*/'/]+", chunk)
        is_known_unit = re.fullmatch(r"\d+[*xX]?\d*(GM|ML|KG|MG|L|TAB|CAP|UNIT|FT|IN)?", chunk, re.IGNORECASE)
        is_feet_inches = re.fullmatch(r"\d{1,3}['\"]", chunk)  # matches 6', 8", etc.
        
        if is_soft_descriptor or is_known_unit or is_feet_inches:
            row['Item Description'] = desc + ' ' + chunk
            row['Batch'] = ''
    
    return row



df = df.apply(clean_batch_and_description, axis=1)

# 4. Clean columns
df['Free'] = df['Free'].fillna(0)

# Forward fill Date
df['Date'] = df['Date'].fillna(method='ffill')

# Convert types
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
for col in ['Qty.', 'Free', 'Rate', 'Amount']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Optional: Reset index and strip strings
df = df.reset_index(drop=True)
df['Item Description'] = df['Item Description'].str.strip()
df['Batch'] = df['Batch'].str.strip()
df['Bill'] = df['Bill'].astype(str).str.strip()
df['Date'] = df['Date'].dt.strftime('%d-%m-%Y')  # Optional formatting

# Save to cleaned file



C:\Users\pranj\AppData\Local\Temp\ipykernel_16696\2816488763.py:74: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Date'] = df['Date'].fillna(method='ffill')


In [18]:
df.to_excel('project_data_202204_202303_cleaned.xlsx', index=False) 